# Activite Pratique N°2 - Partie 1 : Introduction au RAG

Un systeme **RAG (Retrieval-Augmented Generation)** combine deux mecanismes complementaires :

- la **recherche d'information** dans une base documentaire ;
- la **generation de reponse** par un modele de langage.

Dans ce notebook, nous construisons un pipeline RAG simple et pedagogique capable de : charger un ou plusieurs PDF, extraire leur texte, decouper ce texte en chunks, indexer ces chunks dans **Chroma**, retrouver les passages pertinents pour une question, puis generer une reponse avec **OpenAI**.

## 1. Initialisation

Cette section prepare l'environnement de travail : imports, chargement des variables d'environnement et verification de la cle API OpenAI.

In [ ]:
import os
import textwrap
import warnings
from pathlib import Path
from uuid import uuid4

import tiktoken
from dotenv import load_dotenv
from pypdf import PdfReader

from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Cette alerte n'est pas bloquante, mais elle peut apparaitre avec certaines
# versions de LangChain utilisees avec Python 3.14.
warnings.filterwarnings(
    "ignore",
    message="Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.",
)

DATA_DIR = Path("data")
CHROMA_DIR = Path("chroma_db")
CHROMA_DIR.mkdir(exist_ok=True)


In [ ]:
# Chargement du fichier .env a la racine du projet.
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise EnvironmentError(
        "La variable OPENAI_API_KEY est introuvable. Ajoutez-la dans le fichier .env a la racine du projet."
    )

# Le notebook accepte un ou plusieurs PDF dans le dossier data/.
PDF_PATHS = sorted(DATA_DIR.glob("*.pdf"))
if not PDF_PATHS:
    raise FileNotFoundError(
        "Aucun fichier PDF n'a ete trouve dans le dossier 'data'. Ajoutez par exemple 'data/document.pdf'."
    )

print("Cle OpenAI chargee avec succes.")
print(f"Nombre de PDF detectes : {len(PDF_PATHS)}")
for pdf_path in PDF_PATHS:
    print(f"- {pdf_path}")


## 2. Chargement des documents

Nous lisons les fichiers PDF locaux avec **pypdf**, puis nous preparons deux representations utiles :

- le texte brut d'un PDF pour afficher un extrait ;
- une liste de `Document` LangChain, avec metadonnees (`source`, `page`), utile pour l'indexation.

In [ ]:
def clean_text(text: str) -> str:
    """Nettoie grossierement le texte pour supprimer les espaces inutiles."""
    return " ".join(text.split())


def extract_text_from_pdf(pdf_path: Path) -> str:
    """Extrait tout le texte d'un PDF et retourne une seule grande chaine."""
    reader = PdfReader(str(pdf_path))
    pages_text = []

    for page in reader.pages:
        page_text = clean_text(page.extract_text() or "")
        if page_text:
            pages_text.append(page_text)

    return "\n\n".join(pages_text)


def load_pdf_pages(pdf_paths: list[Path]) -> list[Document]:
    """Charge les PDF page par page sous forme de documents LangChain."""
    documents = []

    for pdf_path in pdf_paths:
        reader = PdfReader(str(pdf_path))
        for page_number, page in enumerate(reader.pages, start=1):
            page_text = clean_text(page.extract_text() or "")

            if not page_text:
                continue

            documents.append(
                Document(
                    page_content=page_text,
                    metadata={"source": pdf_path.name, "page": page_number},
                )
            )

    return documents


raw_text = extract_text_from_pdf(PDF_PATHS[0])
page_documents = load_pdf_pages(PDF_PATHS)

if not page_documents:
    raise ValueError("Aucun texte exploitable n'a ete extrait des PDF. Verifiez le contenu des fichiers.")

print(f"Longueur du texte extrait : {len(raw_text):,} caracteres")
print(f"Nombre de documents de base (pages non vides) : {len(page_documents)}")
print("\nExtrait du texte :\n")
print(textwrap.shorten(raw_text, width=1200, placeholder=" ..."))


## 3. Indexing

L'etape d'indexation transforme les documents en morceaux de taille raisonnable, puis calcule leurs embeddings pour les stocker dans une base vectorielle **Chroma**.

In [ ]:
def count_tokens(text: str, model: str = "gpt-4o-mini") -> int:
    """Compte le nombre de tokens pour avoir un indicateur simple de taille."""
    try:
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        encoding = tiktoken.get_encoding("cl100k_base")
    return len(encoding.encode(text))


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", " ", ""],
)

chunked_documents = text_splitter.split_documents(page_documents)

if not chunked_documents:
    raise ValueError("Le decoupage n'a produit aucun chunk. Verifiez le texte extrait.")

token_counts = [count_tokens(doc.page_content) for doc in chunked_documents]

print(f"Nombre total de chunks : {len(chunked_documents)}")
print(f"Nombre moyen de tokens par chunk : {sum(token_counts) / len(token_counts):.1f}")

for index, chunk in enumerate(chunked_documents[:2], start=1):
    print(f"\n--- Chunk {index} ---")
    print(f"Metadonnees : {chunk.metadata}")
    print(textwrap.shorten(chunk.page_content, width=700, placeholder=" ..."))


In [ ]:
# Modele d'embeddings OpenAI recommande pour cette activite.
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# Un nom de collection unique evite de melanger plusieurs executions du notebook.
collection_name = f"tp2_rag_{uuid4().hex[:8]}"

vector_store = Chroma.from_documents(
    documents=chunked_documents,
    embedding=embedding_model,
    collection_name=collection_name,
    persist_directory=str(CHROMA_DIR),
)
vector_store.persist()

print("Base vectorielle Chroma creee avec succes.")
print(f"Collection active : {collection_name}")
print(f"Dossier de persistance : {CHROMA_DIR.resolve()}")


## 4. Retrieval

Nous creons maintenant un **retriever** qui va recuperer les chunks les plus pertinents pour une question donnee. Ici, nous demandons les **4 passages** les plus proches (`k=4`).

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

first_question = "Quel est le sujet principal du document ?"
retrieved_docs = retriever.invoke(first_question)

print(f"Question test : {first_question}")
print(f"Nombre de chunks recuperes : {len(retrieved_docs)}")

for index, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Chunk recupere {index} ---")
    print(f"Metadonnees : {doc.metadata}")
    print(textwrap.shorten(doc.page_content, width=700, placeholder=" ..."))


## 5. Generation

Apres la recherche, nous construisons un prompt contenant le contexte retrouve, puis nous demandons au modele **gpt-4o-mini** de generer une reponse en francais.

In [ ]:
def build_context(docs: list[Document]) -> str:
    """Concatene les chunks recuperes pour construire le contexte du prompt."""
    context_parts = []

    for index, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "inconnu")
        page = doc.metadata.get("page", "?")
        context_parts.append(
            f"[Extrait {index} | source={source} | page={page}]\n{doc.page_content}"
        )

    return "\n\n".join(context_parts)


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt_template = ChatPromptTemplate.from_template(
    """Vous etes un assistant pedagogique specialise en analyse documentaire.
Repondez uniquement a partir du contexte fourni.
Si l'information n'apparait pas dans le contexte, dites-le explicitement sans inventer.

Contexte :
{context}

Question :
{question}

Reponse en francais, claire et structuree :
"""
)

context = build_context(retrieved_docs)
messages = prompt_template.format_messages(context=context, question=first_question)
generated_response = llm.invoke(messages).content

print("Reponse generee :\n")
print(generated_response)


## 6. Pipeline complet

Nous encapsulons maintenant toutes les etapes dans une fonction `rag_pipeline(question, retriever, llm)` afin de reutiliser facilement le systeme.

In [ ]:
def rag_pipeline(question: str, retriever, llm) -> dict:
    """Execute un pipeline RAG simple : retrieval + generation."""
    retrieved_documents = retriever.invoke(question)
    context = build_context(retrieved_documents)

    messages = prompt_template.format_messages(
        context=context,
        question=question,
    )
    answer = llm.invoke(messages).content

    return {
        "question": question,
        "answer": answer,
        "documents": retrieved_documents,
        "context": context,
    }


pipeline_result = rag_pipeline("Quels sont les points importants ?", retriever, llm)

print(f"Question : {pipeline_result['question']}")
print("\nReponse :\n")
print(pipeline_result["answer"])
print(f"\nNombre de documents recuperes : {len(pipeline_result['documents'])}")


## 7. Tests

Nous testons le pipeline sur plusieurs questions afin d'observer son comportement sur des formulations differentes.

In [ ]:
test_questions = [
    "Quel est le sujet principal du document ?",
    "Quels sont les points importants ?",
    "Explique la conclusion du document.",
]

test_results = []

for question in test_questions:
    result = rag_pipeline(question, retriever, llm)
    test_results.append(result)

    print("=" * 100)
    print(f"Question : {question}")
    print("\nReponse :")
    print(result["answer"])
    print("\nSources utilisees :")
    for doc in result["documents"]:
        print(f"- {doc.metadata.get('source')} | page {doc.metadata.get('page')}")


## 8. Evaluation

Cette evaluation reste volontairement simple et qualitative. L'objectif est surtout de verifier que :

- des chunks pertinents sont bien recuperes ;
- la reponse generee reste exploitable ;
- le systeme evite, autant que possible, d'inventer des informations absentes du contexte.

In [ ]:
def qualitative_assessment(answer: str, docs: list[Document]) -> str:
    """Produit une appreciation simple pour un usage pedagogique."""
    if not docs:
        return "Aucun chunk n'a ete retrouve : il faut verifier le retriever ou les embeddings."
    if not answer.strip():
        return "La reponse est vide : la phase de generation a echoue."
    if "pas dans le contexte" in answer.lower() or "information n'est pas disponible" in answer.lower():
        return "Le modele reste prudent, ce qui limite les hallucinations."
    if len(answer) < 80:
        return "La reponse est tres concise : elle peut etre correcte, mais meriterait souvent plus de detail."
    return "La reponse semble exploiter le contexte retrouve. Une verification humaine reste recommandee."


def evaluate_rag(questions: list[str], retriever, llm) -> list[dict]:
    """Execute le pipeline sur plusieurs questions et regroupe les resultats."""
    evaluation_results = []

    for question in questions:
        result = rag_pipeline(question, retriever, llm)
        evaluation_results.append(
            {
                "question": question,
                "answer": result["answer"],
                "num_docs": len(result["documents"]),
                "assessment": qualitative_assessment(result["answer"], result["documents"]),
            }
        )

    return evaluation_results


evaluation_results = evaluate_rag(test_questions, retriever, llm)

print("Evaluation qualitative du pipeline RAG\n")
for item in evaluation_results:
    print("=" * 100)
    print(f"Question : {item['question']}")
    print(f"Nombre de documents recuperes : {item['num_docs']}")
    print(f"Appreciation : {item['assessment']}")
    print("\nReponse :")
    print(item["answer"])


## Conclusion

Dans cette premiere partie, nous avons construit un pipeline **RAG** simple mais complet dans un notebook Jupyter :

- chargement de documents PDF ;
- extraction et nettoyage du texte ;
- decoupage en chunks avec `RecursiveCharacterTextSplitter` ;
- creation des embeddings avec `text-embedding-3-small` ;
- stockage dans une base vectorielle **Chroma** ;
- recuperation des passages pertinents ;
- generation des reponses avec `gpt-4o-mini` ;
- evaluation qualitative sur quelques questions de test.

Cette version reste volontairement simple. Ses limites principales sont la qualite parfois imparfaite de l'extraction PDF, le choix fixe de `k=4`, l'absence de reranking et une evaluation encore essentiellement qualitative. Pour la **Partie 2**, une amelioration naturelle consiste a encapsuler ce pipeline dans une application **Streamlit** afin de construire un chatbot RAG interactif, plus ergonomique et plus demonstratif.